In [ ]:
%load_ext autoreload
%autoreload 2

## 1. Imports

In [2]:
import os
import sys

sys.path.append("..")
sys.path.append("./stylegan3")
sys.path.append("./hyperstyle")

import random

import numpy as np
import torch
from tqdm import tqdm

import wandb
from src.costs.lse import MLPLSECost
from src.models.gmm_based import GMMEOT
from src.plotting.distributions import plot_swiss_roll
from src.plotting.parameters import (
    plot_A_parameters,
    plot_B_parameters,
    plot_Z_parameters,
)
from src.samplers.from_dataset import DatasetSampler
from src.samplers.primary import StandardNormalSampler, SwissRollSampler
from src.utils.discrete_ot import OTPlanSampler
from src.utils.paired import generate_paired_data, get_GT_points, get_paired_sampler
from src.utils.train import compute_loss, update_average

In [3]:
device = torch.device(f"cuda:{torch.cuda.current_device()}" if torch.cuda.is_available() else "cpu")
device

device(type='cuda', index=0)

In [4]:
# torch.set_default_device(device)
# dtype = torch.float64
# torch.torch.set_default_dtype(dtype)

## 2. Config

In [5]:
from configs.gmm_based.cost import MLPLSECostConfig
from configs.gmm_based.dataset import DatasetConfig, MiniBatchConfig
from configs.gmm_based.optimizer import OptPairedConfig, OptUnpairedConfig
from configs.gmm_based.train import TrainConfig

In [6]:
# Data
# Q_X_UNPAIRED_SAMPLES = 16000 # 1024
# R_Y_UNPAIRED_SAMPLES = 16000 # 1024
# P_XY_PAIRED_SAMPLES = 16000 # 128

# Optimizer
LR_PAIRED = 3e-4
LR_UNPAIRED = 3e-4

# Sampler
PAIRED_BATCH_SIZE = 1024
UNPAIRED_BATCH_SIZE = 1024

# Train
X_DIM = 512
MAX_STEPS = 10000
INIT_BY_SAMPLES = True

# Potential
Y_DIM = 512
N_POTENTIALS = 1

# Cost
M_POTENTIALS = 4
LOG_V_M_HIDDEN_CHANNELS = [Y_DIM, Y_DIM]
B_M_HIDDEN_CHANNELS = [Y_DIM, Y_DIM, Y_DIM]

In [7]:
# dataset_config = DatasetConfig(
#     P_XY_paired=P_XY_PAIRED_SAMPLES, Q_X_unpaired=Q_X_UNPAIRED_SAMPLES, R_Y_unpaired=R_Y_UNPAIRED_SAMPLES
# )
minibatch_config = MiniBatchConfig()

cost_config = MLPLSECostConfig(
    x_dim=X_DIM,
    y_dim=Y_DIM,
    m_potentials=M_POTENTIALS,
    log_v_m_hidden_channels=LOG_V_M_HIDDEN_CHANNELS,
    b_m_hidden_channels=B_M_HIDDEN_CHANNELS,
)
EXP_META_INFO = (
    f"M_POTENTIALS_{M_POTENTIALS}_"
    + f"LOG_V_M_HIDDEN_CHANNELS_{LOG_V_M_HIDDEN_CHANNELS}_"
    + f"B_M_HIDDEN_CHANNELS_{B_M_HIDDEN_CHANNELS}_"
    + "class_to_class"
)

opt_unpaired_config = OptUnpairedConfig(lr=LR_UNPAIRED)
opt_paired_config = OptPairedConfig(lr=LR_PAIRED)

train_config = TrainConfig(
    steps_to=MAX_STEPS, paired_batch_size=PAIRED_BATCH_SIZE, unpaired_batch_size=UNPAIRED_BATCH_SIZE
)

In [8]:
torch.manual_seed(train_config.seed)
np.random.seed(train_config.seed)
random.seed(train_config.seed)

## 3. Create data and samplers

In [12]:
%%bash
module load compilers/gcc-12.2.0

In [13]:
import torchvision.transforms as transforms
from hyperstyle.utils.model_utils import load_model, load_generator
from hyperstyle.utils.inference_utils import run_inversion
import pprint
from pathlib import Path

In [14]:
EXPERIMENT_ARGS = {
    "model_path": "./hyperstyle/pretrained_models/hyperstyle_afhq_wild.pt",
    "w_encoder_path": "./hyperstyle/pretrained_models/afhq_wild_w_encoder.pt",
    "image_path": "./stylegan/images/afhq_wild_image.jpg",
    "transform": transforms.Compose(
        [transforms.Resize((256, 256)), transforms.ToTensor(), transforms.Normalize([0.5, 0.5, 0.5], [0.5, 0.5, 0.5])]
    ),
}

In [15]:
#@title Load HyperStyle Model { display-mode: "form" } 
model_path = EXPERIMENT_ARGS['model_path']
net, opts = load_model(model_path, update_opts={"w_encoder_checkpoint_path": EXPERIMENT_ARGS['w_encoder_path']})
print('Model successfully loaded!')
pprint.pprint(vars(opts))

/trinity/home/m.persiyanov/miniconda3/envs/style/lib/python3.10/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/trinity/home/m.persiyanov/miniconda3/envs/style/lib/python3.10/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet34_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet34_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Loading HyperStyle from checkpoint: ./hyperstyle/pretrained_models/hyperstyle_afhq_wild.pt
Loading pretrained W encoder...
Using WEncoder
Loading WEncoder from checkpoint: ./hyperstyle/pretrained_models/afhq_wild_w_encoder.pt
Model successfully loaded!
{'batch_size': 8,
 'board_interval': 50,
 'checkpoint_path': './hyperstyle/pretrained_models/hyperstyle_afhq_wild.pt',
 'dataset_type': 'afhq_wild_hypernet',
 'device': 'cuda:0',
 'encoder_type': 'SharedWeightsHyperNetResNet',
 'exp_dir': '',
 'id_lambda': 0,
 'image_interval': 100,
 'input_nc': 6,
 'l2_lambda': 1.0,
 'layers_to_tune': '5,6,8,9,11,12,14,15,17,18,20,21',
 'learning_rate': 0.0001,
 'load_w_encoder': True,
 'lpips_lambda': 0.8,
 'max_steps': 500000,
 'max_val_batches': 150,
 'moco_lambda': 0.5,
 'n_hypernet_outputs': 23,
 'n_iters_per_batch': 5,
 'optim_name': 'ranger',
 'output_size': 512,
 'save_interval': 10000,
 'start_from_w_plus': False,
 'stylegan_weights': '',
 'test_batch_size': 8,
 'test_workers': 8,
 'train_decod

In [16]:
def get_latent_and_weight_deltas(inputs, net, opts):
    opts.resize_outputs = False
    opts.n_iters_per_batch = 5
    with torch.no_grad():
        _, latent, weights_deltas, _ = run_inversion(inputs.to("cuda").float(), net, opts)
    weights_deltas = [w[0] if w is not None else None for w in weights_deltas]
    return latent, weights_deltas

In [17]:
# from src.utils.dataset.AFHQ import get_latent_vectors
from src.samplers.base import TensorLabeledSampler, PairedLabeledSampler
from PIL import Image

In [18]:
SPLIT = "train"

In [19]:
img_transforms = EXPERIMENT_ARGS['transform']

path = Path('./stylegan3/data/train')

In [20]:
# all_vecs = []
# for image_path in tqdm(list(path.rglob('*'))):
#     if image_path.is_file():
#         # print(f'Working on {image_path}...')
#         original_image = Image.open(image_path)
#         original_image = original_image.convert("RGB")
#         input_image = img_transforms(original_image)
#         # get the weight deltas for each image
#         result_vec, weights_deltas = get_latent_and_weight_deltas(input_image.unsqueeze(0), net, opts)
#         all_vecs.append(result_vec[0])

In [21]:
latent_labels = []
for image_path in path.rglob('*'):
    if image_path.is_file():
        latent_labels.append(image_path)

In [22]:
import pickle

In [23]:
# with open('latents.pickle', 'wb') as handle:
#     pickle.dump(all_vecs, handle, protocol=pickle.HIGHEST_PROTOCOL)

In [24]:
with open('latents.pickle', 'rb') as f:
    all_vecs = pickle.load(f)

In [25]:
all_vecs = all_vecs[:-3]

In [26]:
# with open('latent_labels.pickle', 'wb') as handle:
#     pickle.dump(list(path.rglob('*')), handle, protocol=pickle.HIGHEST_PROTOCOL)

In [27]:
X_unpaired = torch.stack([vec[0][0] for vec in all_vecs])
X_unpaired_labels = latent_labels

In [28]:
Y_unpaired = X_unpaired.clone()
Y_unpaired_labels = latent_labels

In [29]:
# X_cat, y_cat = get_latent_vectors("./stylegan3/stylegan3_latents", "cat", SPLIT, False, device=device)
# assert len(X_cat) == len(y_cat)
# len(X_cat)
# X_wild, y_wild = get_latent_vectors("./stylegan3/stylegan3_latents", "wild", SPLIT, False, device=device)
# assert len(X_wild) == len(y_wild)
# len(X_wild)
# X_dog, y_dog = get_latent_vectors("./stylegan3/stylegan3_latents", "dog", SPLIT, False, device=device)
# assert len(X_dog) == len(y_dog)
# len(X_dog)

In [30]:
X_cat = torch.stack([x for x, label in zip(X_unpaired, X_unpaired_labels) if "cat" in str(label)])
y_cat = [label for x, label in zip(X_unpaired, X_unpaired_labels) if "cat" in str(label)]
assert len(X_cat) == len(y_cat)
print(len(X_cat))
X_wild = torch.stack([x for x, label in zip(X_unpaired, X_unpaired_labels) if "wild" in str(label)])
y_wild = [label for _, label in zip(X_unpaired, X_unpaired_labels) if "wild" in str(label)]
assert len(X_wild) == len(y_wild)
print(len(X_wild))
X_dog = torch.stack([x for x, label in zip(X_unpaired, X_unpaired_labels) if "dog" in str(label)])
y_dog = [label for _, label in zip(X_unpaired, X_unpaired_labels) if "dog" in str(label)]
assert len(X_dog) == len(y_dog)
print(len(X_dog))

5065
4593
4678


In [31]:
# X_unpaired = torch.cat([X_cat, X_wild, X_dog])
# X_unpaired_labels = y_cat + y_wild + y_dog

In [32]:
# Y_unpaired = X_unpaired.clone()
# Y_unpaired_labels = deepcopy(X_unpaired_labels)

### Making paired data

In [33]:
from pathlib import Path
import re

In [34]:
def read_labels_from_txt(filename: str):
    # Define the substrings to be replaced and the replacement string
    substrings_to_replace = {"foxes", "lion", "wolf", "tiger", "leopard"}
    replacement_string = "wild"

    # Create a regular expression pattern that matches any of the substrings
    pattern = re.compile(r'\b(' + '|'.join(map(re.escape, substrings_to_replace)) + r')\b')

    labels = []
    with open(filename, 'r') as file:
        for line in file:
            parts = line.strip().split()
            # Perform the replacement
            modified_path = pattern.sub(replacement_string, parts[0])
            labels.append(Path(modified_path))
    return labels

In [35]:
source_labels = read_labels_from_txt('notebooks/source_balanced_list.txt')
target_labels = read_labels_from_txt('notebooks/target_balanced_list.txt')

In [36]:
label_to_ind = dict(zip(X_unpaired_labels, range(len(X_unpaired_labels))))

In [37]:
X_paired_labels = [Path("stylegan3/data/train") / name for name in source_labels]
Y_paired_labels = [Path("stylegan3/data/train") / name for name in target_labels]

In [38]:
X_paired_train = torch.stack([X_unpaired[label_to_ind[name]] for name in X_paired_labels])
Y_paired_train = torch.stack([Y_unpaired[label_to_ind[name]] for name in Y_paired_labels])

In [39]:
DATA_METHOD = "sinkhorn"
DATA_COST_FUNCTION = "l2"
otp_sampler = OTPlanSampler(DATA_METHOD, cost_function=DATA_COST_FUNCTION)

In [40]:
# DATA_DIR = "./data/AFHQ"
# NUM_SAMPLES_FOR_EACH_TRANSFER = 1000
# file_postfix = "cat->wild"
# MINI_BATCH_SIZE = 128

In [41]:
def make_pairs(
    X: torch.Tensor, X_labels: list[str], Y: torch.Tensor, Y_labels: list[str]
) -> tuple[torch.Tensor, list[str], torch.Tensor, list[str]]:
    X_paired_list, X_labels_list = [], []
    Y_paired_list, Y_labels_list = [], []

    p = otp_sampler.get_map(X, Y)

    for i, row in enumerate(p):
        target_ind = np.argmax(row)
        X_paired_list.append(X[i])
        X_labels_list.append(X_labels[i])
        Y_paired_list.append(Y[target_ind])
        Y_labels_list.append(Y_labels[target_ind])

    return X_paired_list, X_labels_list, Y_paired_list, Y_labels_list

In [42]:
# X1, X_1_labels, Y1, Y_1_labels = make_pairs(X_paired_train[:1000], X_paired_labels[:1000], Y_paired_train[:1000], Y_paired_labels[:1000])
# X2, X_2_labels, Y2, Y_2_labels = make_pairs(X_paired_train[1000:2000], X_paired_labels[1000:2000], Y_paired_train[1000:2000], Y_paired_labels[1000:2000])
# X3, X_3_labels, Y3, Y_3_labels = make_pairs(X_paired_train[2000:], X_paired_labels[2000:], Y_paired_train[2000:], Y_paired_labels[2000:])

In [43]:
# X_res, Y_res = [], []
# X_labels, Y_labels = [], []

# for x, y, x_label, y_label in zip([X1, X2, X3], [Y1, Y2, Y3], [X_1_labels, X_2_labels, X_3_labels], [Y_1_labels, Y_2_labels, Y_3_labels]):
#     X_res.extend(x)
#     Y_res.extend(y)
#     X_labels.extend(x_label)
#     Y_labels.extend(y_label)

In [45]:
from_data = [X_cat, X_wild, X_dog]
from_labels = [y_cat, y_wild, y_dog]

to_data = [X_wild, X_dog, X_cat]
to_labels = [y_wild, y_dog, y_cat]

X_res, X_labels = [], []
Y_res, Y_labels = [], []
for src_data, src_labels, trgt_data, trgt_labels in zip(from_data, from_labels, to_data, to_labels):
    _X_paired, _X_labels, _Y_paired, _Y_labels = make_pairs(src_data, src_labels, trgt_data, trgt_labels)
    X_res.extend(_X_paired)
    Y_res.extend(_Y_paired)
    X_labels.extend(_X_labels)
    Y_labels.extend(_Y_labels)

In [46]:
X_paired_train = torch.stack(X_res)
X_paired_labels = X_labels
Y_paired_train = torch.stack(Y_res)
Y_paired_labels = X_labels

In [47]:
# NETWORK_PKL = NETWORK_PKL = "./stylegan3/pretrained/stylegan3.pkl"

In [48]:
# import dnnlib
# import legacy
# import PIL.Image

In [49]:
# with dnnlib.util.open_url(NETWORK_PKL) as fp:
#     G = legacy.load_network_pkl(fp)["G_ema"].requires_grad_(False).to(device) 

In [50]:
!module load compilers/gcc-12.2.0

In [51]:
# latent_path = os.path.join("./stylegan3/stylegan3_latents/train", X_labels[0]).replace(".png", "_projected_w.npz")

# z = torch.from_numpy(np.load(latent_path)["w"][0]).to(device)

In [52]:
import PIL.Image
import matplotlib.pyplot as plt
from hyperstyle.utils.common import tensor2im

%matplotlib notebook

In [53]:
img_path = Path('stylegan3/data/train/cat/flickr_cat_000684.png')
img_latent_ind = label_to_ind[img_path]

In [54]:
print(f'Working on {os.path.basename(img_path)}...')
original_image = Image.open(img_path)
original_image = original_image.convert("RGB")
input_image = img_transforms(original_image).cuda()

Working on flickr_cat_000684.png...


In [55]:
result_image = net.decoder([X_unpaired[img_latent_ind].repeat(1, 16, 1)],
                              weights_deltas=None,
                              randomize_noise=False,
                              input_is_latent=True)[0]

In [56]:
# Set up the plot to display images side by side
n_images = 1
fig, axes = plt.subplots(n_images, 2, figsize=(10, n_images * 3))

axes[0].imshow(tensor2im(input_image))
axes[0].axis('off')  # Hide axis

# Plot second image from list2
axes[1].imshow(tensor2im(result_image[0]))
axes[1].axis('off')  # Hide axis
    

# Display the plot
plt.tight_layout()
plt.savefig("1.png")
plt.show()

<IPython.core.display.Javascript object>

In [57]:
original_images = []
generated_images = []
num_images = 4

for X, X_label in tqdm(zip(Y_paired_train[:num_images], Y_paired_labels[:num_images])):
    # Load target image.
    target_pil = PIL.Image.open(X_label).convert("RGB")
    w, h = target_pil.size
    s = min(w, h)
    target_pil = target_pil.crop(((w - s) // 2, (h - s) // 2, (w + s) // 2, (h + s) // 2))
    target_pil = target_pil.resize((256, 256), PIL.Image.LANCZOS)
    target_uint8 = np.array(target_pil, dtype=np.uint8)
    original_images.append(target_uint8)

    synth_image = net.decoder([X.repeat(1, 16, 1)],
                              weights_deltas=None,
                              randomize_noise=False,
                              input_is_latent=True)[0]
    # noise_mode = "const"
    # trunc = 1
    # seed = 10
    # img = G(X.unsqueeze(0), truncation_psi=trunc, noise_mode=noise_mode, c=None)
    # # permute to [b, 512, 512, c] and scale to 0-255
    # img = (img.permute(0, 2, 3, 1) * 127.5 + 128).clamp(0, 255).to(torch.uint8)
    # generated_images.append(img.squeeze(0).cpu().numpy())
    # synth_image = G.synthesis(X.repeat(1, 16, 1), noise_mode="const")
    synth_image = (synth_image + 1) * (255 / 2)
    synth_image = synth_image.permute(0, 2, 3, 1).clamp(0, 255).to(torch.uint8)[0].cpu().numpy()
    generated_images.append(synth_image)

4it [00:00, 32.95it/s]


In [58]:
# Set up the plot to display images side by side
n_images = len(original_images)
fig, axes = plt.subplots(n_images, 2, figsize=(10, n_images * 3))

# Loop through images and plot them side by side
for i in range(n_images):
    # Plot first image from list1
    axes[i, 0].imshow(original_images[i])
    axes[i, 0].axis('off')  # Hide axis

    # Plot second image from list2
    axes[i, 1].imshow(generated_images[i])
    axes[i, 1].axis('off')  # Hide axis

# Display the plot
# plt.tight_layout()
plt.savefig("1.png")
# plt.show()

<IPython.core.display.Javascript object>

In [59]:
pd_train_sampler = PairedLabeledSampler(X_paired_train, X_paired_labels, Y_paired_train, Y_paired_labels)

In [60]:
usd_sampler = TensorLabeledSampler(X_unpaired, X_unpaired_labels, device=device)
utd_sampler = TensorLabeledSampler(Y_unpaired, Y_unpaired_labels, device=device)

In [61]:
# X_unpaired_test = X_sampler.sample(dataset_config.P_XY_paired)
# Y_unpaired_test = Y_sampler.sample(dataset_config.P_XY_paired)

In [62]:
# if Q_X_unpaired > 0:
#     source_data = X_sampler.sample(dataset_config.Q_X_unpaired)
#     usd_sampler = DatasetSampler(source_data, device=device) # usd - unpaired source data
# else:
#     usd_sampler = DatasetSampler(X_paired_train, device=device)

# if dataset_config.R_Y_unpaired > 0:
#     target_data = Y_sampler.sample(dataset_config.R_Y_unpaired)
#     utd_sampler = DatasetSampler(target_data, device=device) # utd - unpaired target data
# else:
#     utd_sampler = DatasetSampler(Y_paired_train, device=device)

## 4. Model initialization

In [63]:
cost = MLPLSECost(
    log_v_m_hidden_channels=LOG_V_M_HIDDEN_CHANNELS,
    b_m_hidden_channels=B_M_HIDDEN_CHANNELS,
    x_dim=X_DIM,
    y_dim=Y_DIM,
    m_potentials=M_POTENTIALS,
).to(device)

In [64]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class MLP_UNet(nn.Module):
    def __init__(self, input_dim=512):
        super(MLP_UNet, self).__init__()
        
        # Contracting Path (Encoder)
        self.enc1 = nn.Sequential(
            nn.Linear(input_dim, 512),
            nn.ReLU(),
            nn.Linear(512, 512),
            nn.ReLU()
        )
        self.down1 = nn.Linear(512, 256)
        
        self.enc2 = nn.Sequential(
            nn.Linear(256, 256),
            nn.ReLU(),
            nn.Linear(256, 256),
            nn.ReLU()
        )
        self.down2 = nn.Linear(256, 128)
        
        self.enc3 = nn.Sequential(
            nn.Linear(128, 128),
            nn.ReLU(),
            nn.Linear(128, 128),
            nn.ReLU()
        )
        self.down3 = nn.Linear(128, 64)
        
        self.enc4 = nn.Sequential(
            nn.Linear(64, 64),
            nn.ReLU(),
            nn.Linear(64, 64),
            nn.ReLU()
        )
        self.down4 = nn.Linear(64, 32)
        
        # Bottleneck
        self.bottleneck = nn.Sequential(
            nn.Linear(32, 32),
            nn.ReLU(),
            nn.Linear(32, 32),
            nn.ReLU()
        )
        
        # Expanding Path (Decoder)
        self.up1 = nn.Linear(32, 64)
        self.dec1 = nn.Sequential(
            nn.Linear(128, 64),  # 64 (up) + 64 (skip) = 128
            nn.ReLU(),
            nn.Linear(64, 64),
            nn.ReLU()
        )
        
        self.up2 = nn.Linear(64, 128)
        self.dec2 = nn.Sequential(
            nn.Linear(256, 128),  # 128 (up) + 128 (skip) = 256
            nn.ReLU(),
            nn.Linear(128, 128),
            nn.ReLU()
        )
        
        self.up3 = nn.Linear(128, 256)
        self.dec3 = nn.Sequential(
            nn.Linear(512, 256),  # 256 (up) + 256 (skip) = 512
            nn.ReLU(),
            nn.Linear(256, 256),
            nn.ReLU()
        )
        
        self.up4 = nn.Linear(256, 512)
        self.dec4 = nn.Sequential(
            nn.Linear(1024, 512),  # 512 (up) + 512 (skip) = 1024
            nn.ReLU(),
            nn.Linear(512, 512),
            nn.ReLU()
        )
        
        # Output layer
        self.output = nn.Linear(512, input_dim)
        
    def forward(self, x):
        # Encoder
        c1 = self.enc1(x)
        p1 = F.relu(self.down1(c1))
        
        c2 = self.enc2(p1)
        p2 = F.relu(self.down2(c2))
        
        c3 = self.enc3(p2)
        p3 = F.relu(self.down3(c3))
        
        c4 = self.enc4(p3)
        p4 = F.relu(self.down4(c4))
        
        # Bottleneck
        bottleneck = self.bottleneck(p4)
        
        # Decoder with skip connections
        u1 = F.relu(self.up1(bottleneck))
        u1 = torch.cat([u1, c4], dim=1)
        u1 = self.dec1(u1)
        
        u2 = F.relu(self.up2(u1))
        u2 = torch.cat([u2, c3], dim=1)
        u2 = self.dec2(u2)
        
        u3 = F.relu(self.up3(u2))
        u3 = torch.cat([u3, c2], dim=1)
        u3 = self.dec3(u3)
        
        u4 = F.relu(self.up4(u3))
        u4 = torch.cat([u4, c1], dim=1)
        u4 = self.dec4(u4)
        
        # Output
        # out = torch.sigmoid(self.output(u4))
        out = self.output(u4)
        return out

In [65]:
u = MLP_UNet(input_dim=X_DIM)

In [66]:
from src.costs.lse import BatchedLSECost

In [67]:
import torch.nn as nn
import torchvision

In [68]:
log_v_m_net = nn.Sequential(
            torchvision.ops.MLP(
                in_channels=X_DIM, hidden_channels=[Y_DIM, Y_DIM, M_POTENTIALS]
            ),
            nn.LogSoftmax(dim=-1),
        ).to(device)

In [69]:
cost = BatchedLSECost(u, log_v_m_net, m_potentials=M_POTENTIALS)

In [70]:
import gc
torch.cuda.empty_cache()
gc.collect()

3955

In [71]:
model = GMMEOT(
    y_dim=Y_DIM,
    n_potentials=N_POTENTIALS,
    cost=cost,
).to(device)

if INIT_BY_SAMPLES:
    model.init_a_by_samples(usd_sampler.sample(N_POTENTIALS)[0])

In [72]:
# For EMA update
if train_config.ema_update:
    model_copy = GMMEOT(
    y_dim=Y_DIM,
    n_potentials=N_POTENTIALS,
    cost=cost,
).to(device)

## 5. Optimizers initialization

In [73]:
# unpaired_params_to_update = [model._log_w_n, model._a_n, model._log_A_n]

# D_opt_unpaired = torch.optim.Adam(unpaired_params_to_update, **opt_unpaired_config.model_dump())

In [74]:
# D_opt_paired = torch.optim.Adam(model.cost.parameters(), **opt_paired_config.model_dump())

In [75]:
optimizer =  torch.optim.Adam(model.parameters(), lr=3e-4)
# optimizer =  torch.optim.Adam(model.cost.parameters(), lr=1e-3)

In [76]:
# TODO: refactor this config
EXP_NAME = (
    "GMMEOT_AFHQ_"
    # + f"P_XY_PAIRED_{P_XY_paired}_"
    # + f"Q_X_UNPAIRED_{Q_X_unpaired}_"
    # + f"R_Y_UNPAIRED_{R_Y_unpaired}_"
    + f"LR_PAIRED_{opt_paired_config.lr}_"
    + f"LR_UNPAIRED_{opt_unpaired_config.lr}_"
    + f"MINIBATCH_COST_{minibatch_config.cost_function}_"
    + EXP_META_INFO
)
OUTPUT_PATH = "../checkpoints/{}".format(EXP_NAME)

config = dict(
    # X_DIM=dataset_config.x_dim,
    # Y_DIM=dataset_config.y_dim,
    D_LR_PAIRED=opt_paired_config.lr,
    D_LR_UNPAIRED=opt_unpaired_config.lr,
    BATCH_SIZE=train_config.unpaired_batch_size,
    # P_XY_PAIRED_SAMPLES=dataset_config.P_XY_paired,
    # Q_X_UNPAIRED_SAMPLES=dataset_config.Q_X_unpaired,
    # R_Y_UNPAIRED_SAMPLES=dataset_config.R_Y_unpaired,
)

if not os.path.exists(OUTPUT_PATH):
    os.makedirs(OUTPUT_PATH, exist_ok=True)

In [77]:
if train_config.steps_from > 0:
    D_opt_unpaired.load_state_dict(torch.load(os.path.join(OUTPUT_PATH, f"D_opt_unpaired_{train_config.steps_from}.pt")))
    D_opt_paired.load_state_dict(torch.load(os.path.join(OUTPUT_PATH, f"D_opt_paired_{train_config.steps_from}.pt")))

## 6. Model training

In [78]:
starting_points = X_unpaired[:5].to(device) # torch.tensor()# [[-2.0, 0.0], [2.0, 2.0], [0.0, 0.0]])
num_ending_points = 64

In [79]:
# num_starting_points_paired = 5
# indices = random.choices(range(dataset_config.P_XY_paired), k=num_starting_points_paired)
# starting_points_paired = X_paired_train[indices]
# ending_points_paired = Y_paired_train[indices]

In [80]:
# gt_Y_points = get_GT_points(X_sampler, Y_sampler, otp_sampler, starting_points)

In [81]:
# wandb.finish()

In [82]:
wandb.init(name=EXP_NAME, config=config)

for step in tqdm(range(train_config.steps_from, train_config.steps_to)):
    # training loop
    # D_opt_unpaired.zero_grad()
    optimizer.zero_grad()

    X, _ = usd_sampler.sample(train_config.unpaired_batch_size)
    Y, _ = utd_sampler.sample(train_config.unpaired_batch_size)

    output_unpaired = model.compute_unpaired_loss(X.to(device), Y.to(device))
    D_loss_unpaired = output_unpaired["loss"]

    wandb.log({f"Unpaired loss": D_loss_unpaired.item()}, step=step)

    # D_opt_paired.zero_grad()
    X_paired, _, Y_paired, _ = pd_train_sampler.sample(train_config.paired_batch_size)
    
    output_paired = model.compute_paired_loss(X_paired.to(device), Y_paired.to(device))
    D_loss_paired = output_paired["loss"]

    wandb.log({f"Paired loss": D_loss_paired.item()}, step=step)

    D_loss = D_loss_unpaired + D_loss_paired
    D_loss.backward()
    optimizer.step()
    # D_opt_paired.step()
    # D_opt_unpaired.step()

    if train_config.ema_update:
        update_average(model_copy, model, 0.99)
        model = model_copy
    else:
        model = model

    torch.cuda.empty_cache()
    gc.collect()

    with torch.no_grad():
        wandb.log({f"Loss": D_loss}, step=step)
        # wandb.log(
        #     {f"Train paired loss": compute_loss(model, X_paired_train, Y_paired_train, X_paired_train, Y_paired_train)},
        #     step=step,
        # )
        # wandb.log(
        #     {f"Test paired loss": compute_loss(model, X_paired_test, Y_paired_test, X_paired_test, Y_paired_test)},
        #     step=step,
        # )
        # wandb.log(
        #     {f"Test unpaired loss": compute_loss(model, X_unpaired_test, Y_unpaired_test, X_paired_test, Y_paired_test)},
        #     step=step,
        # )

        wandb.log({r"$-f^c(x)$": -output_unpaired["f_c"].mean().item()}, step=step)
        wandb.log({r"$-f(y)$": -output_unpaired["f"].mean().item()}, step=step)
        wandb.log({f"lam_min(A_n)": torch.min(output_unpaired["A_n"])}, step=step)
        wandb.log({f"lam_max(A_n)": torch.max(output_unpaired["A_n"])}, step=step)

        if step % 100 == 0:
            # A_dict = plot_A_parameters(model, log=True)
            # B_dict = plot_B_parameters(model.cost, starting_points, log=True)
            # wandb.log(A_dict | B_dict)
            # Z_dict = plot_Z_parameters(model, starting_points, log=True)
            # wandb.log(A_dict | B_dict | Z_dict)
            # if num_starting_points_paired > 0:
            #     Z_dict = plot_Z_parameters(model, starting_points, starting_points_paired, ending_points_paired, log=True)
            # else:
            #     Z_dict = plot_Z_parameters(model, starting_points, log=True)
            # distr_dict = plot_swiss_roll(
            #     {f"P={P_XY_PAIRED_SAMPLES}, Q={Q_X_UNPAIRED_SAMPLES}, R={R_Y_UNPAIRED_SAMPLES}": model},
            #     X_sampler,
            #     Y_sampler,
            #     X_paired,
            #     Y_paired,
            #     starting_points,
            #     gt_Y_points,
            #     log=True,
            # )
            # wandb.log(A_dict | B_dict | Z_dict | distr_dict)

            torch.save(model.state_dict(), os.path.join(OUTPUT_PATH, f"D_{step}.pt"))
            torch.save(optimizer.state_dict(), os.path.join(OUTPUT_PATH, f"opt_{step}.pt"))

torch.save(model.state_dict(), os.path.join(OUTPUT_PATH, f"D_{MAX_STEPS}.pt"))
# torch.save(D_opt_paired.state_dict(), os.path.join(OUTPUT_PATH, f"D_opt_paired_{MAX_STEPS}.pt"))
# torch.save(D_opt_unpaired.state_dict(), os.path.join(OUTPUT_PATH, f"D_opt_unpaired_{MAX_STEPS}.pt"))

wandb.finish()

wandb: Currently logged in as: muxaujl11110. Use `wandb login --relogin` to force relogin
wandb: Using wandb-core as the SDK backend.  Please refer to https://wandb.me/wandb-core for more information.


100%|█████████████████████████████████████████████████████████████████████████████████| 10000/10000 [40:31<00:00,  4.11it/s]


$-f(y)$,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▂▂▂▂▂▃▃▃▃▅▅▆▆█
$-f^c(x)$,██████████████████████▇▇▇▇▇▆▆▆▅▅▅▄▃▃▂▂▂▁
Loss,█████████████████▇▇▇▇▇▇▇▇▆▆▆▆▆▆▅▅▅▅▄▄▄▄▁
Paired loss,████████████████████▇▇▇▇▇▇▆▆▆▆▅▅▅▅▅▄▃▃▃▁
Unpaired loss,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▂▂▂▂▂▂▂▂▂▃▃▃▃▃▃▄▄▅▅▅▆▆▇██
lam_max(A_n),▁▁▁▁▁▂▂▂▂▂▃▄▄▄▄▅▅▅▅▆▆▆▆▇▇▇▇▇█████████▇▇▇
lam_min(A_n),██▇▇▇▅▅▄▄▄▄▄▃▃▃▃▃▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
$-f(y)$,560457.125
$-f^c(x)$,-520558.8125
Loss,-128685.09375
Paired loss,-168583.40625


In [ ]:
!module load compilers/gcc-12.2.0

In [83]:
os.path.join(OUTPUT_PATH, f"D_{MAX_STEPS}.pt")

'../checkpoints/GMMEOT_AFHQ_LR_PAIRED_0.0003_LR_UNPAIRED_0.0003_MINIBATCH_COST_rotation-v2_M_POTENTIALS_4_LOG_V_M_HIDDEN_CHANNELS_[512, 512]_B_M_HIDDEN_CHANNELS_[512, 512, 512]_class_to_class/D_10000.pt'

In [84]:
load_from_step = 600
model.load_state_dict(torch.load(os.path.join(OUTPUT_PATH, f"D_{load_from_step}.pt"), map_location=device))
# optimizer.load_state_dict(torch.load(os.path.join(OUTPUT_PATH, f"opt_{load_from_step}.pt"), map_location=device))

/tmp/ipykernel_1884364/1229225763.py:2: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load(os.path.join(OUTPUT_PATH, f"D_{load_from_step}.pt"), m

<All keys matched successfully>

In [85]:
X_paired.shape

torch.Size([1024, 512])

In [86]:
Y_paired_labels

[PosixPath('stylegan3/data/train/cat/pixabay_cat_003152.png'),
 PosixPath('stylegan3/data/train/cat/pixabay_cat_003229.png'),
 PosixPath('stylegan3/data/train/cat/pixabay_cat_000973.png'),
 PosixPath('stylegan3/data/train/cat/pixabay_cat_004380.png'),
 PosixPath('stylegan3/data/train/cat/flickr_cat_000768.png'),
 PosixPath('stylegan3/data/train/cat/pixabay_cat_003072.png'),
 PosixPath('stylegan3/data/train/cat/pixabay_cat_004521.png'),
 PosixPath('stylegan3/data/train/cat/flickr_cat_000646.png'),
 PosixPath('stylegan3/data/train/cat/pixabay_cat_003450.png'),
 PosixPath('stylegan3/data/train/cat/pixabay_cat_001659.png'),
 PosixPath('stylegan3/data/train/cat/pixabay_cat_000486.png'),
 PosixPath('stylegan3/data/train/cat/pixabay_cat_003406.png'),
 PosixPath('stylegan3/data/train/cat/pixabay_cat_002734.png'),
 PosixPath('stylegan3/data/train/cat/pixabay_cat_004275.png'),
 PosixPath('stylegan3/data/train/cat/pixabay_cat_003414.png'),
 PosixPath('stylegan3/data/train/cat/flickr_cat_000600.pn

In [87]:
Y_paired_labels[1000:1005]

[PosixPath('stylegan3/data/train/cat/pixabay_cat_003694.png'),
 PosixPath('stylegan3/data/train/cat/pixabay_cat_000505.png'),
 PosixPath('stylegan3/data/train/cat/pixabay_cat_002720.png'),
 PosixPath('stylegan3/data/train/cat/pixabay_cat_000185.png'),
 PosixPath('stylegan3/data/train/cat/flickr_cat_000643.png')]

In [88]:
ind = np.random.randint(0, len(X_paired_train), num_images)
[X_paired_labels[i] for i in ind]

[PosixPath('stylegan3/data/train/cat/pixabay_cat_004469.png'),
 PosixPath('stylegan3/data/train/cat/flickr_cat_000786.png'),
 PosixPath('stylegan3/data/train/dog/pixabay_dog_002826.png'),
 PosixPath('stylegan3/data/train/cat/pixabay_cat_003072.png')]

In [89]:
original_images = []
generated_images = []
num_images = 10

# X_test = X_dog[:num_images]
# y_test = y_dog[:num_images]
# category = "dog"
# X_test = X_cat[:]
# y_test = y_cat[:num_images]
# category = "cat"
# X_test = X_wild[:]
# y_test = y_wild[:num_images]
# category = "wild"
# ind = np.random.randint(0, 3000, num_images)
ind = np.random.randint(0, len(X_paired_train), num_images)
print(ind)
# ind = list(range(2000, 2010))
X_test = [X_paired_train[i] for i in ind]
y_test = [X_paired_labels[i] for i in ind]
print(y_test)
for X, X_label in tqdm(zip(X_test, y_test)):
    # print(X_label.removesuffix('_projected_w'), Y_label.removesuffix('_projected_w'))
    # Load target image.
    target_pil = PIL.Image.open(X_label).convert("RGB")
    input_image = img_transforms(target_pil)
    original_images.append(tensor2im(input_image))
    # w, h = target_pil.size
    # s = min(w, h)
    # target_pil = target_pil.crop(((w - s) // 2, (h - s) // 2, (w + s) // 2, (h + s) // 2))
    # target_pil = target_pil.resize((256, 256), PIL.Image.LANCZOS)
    # target_uint8 = np.array(target_pil, dtype=np.uint8)
    # original_images.append(target_uint8)

    X = model(X[None, :]).squeeze()
    result_image = net.decoder([X.repeat(1, 16, 1).cuda()],
                              weights_deltas=None,
                              randomize_noise=False,
                              input_is_latent=True)[0][0]
    generated_images.append(tensor2im(result_image))
    # noise_mode = "const"
    # trunc = 1
    # seed = 10
    # pred = model(X.unsqueeze(0))
    # img = G(pred, truncation_psi=trunc, noise_mode=noise_mode, c=None)
    # # permute to [b, 512, 512, c] and scale to 0-255
    # img = (img.permute(0, 2, 3, 1) * 127.5 + 128).clamp(0, 255).to(torch.uint8)
    # generated_images.append(img.squeeze(0).cpu().numpy())
    # synth_image = G.synthesis(X.repeat(1, 16, 1), noise_mode="const")
    # synth_image = (synth_image + 1) * (255 / 2)
    # synth_image = synth_image.permute(0, 2, 3, 1).clamp(0, 255).to(torch.uint8)[0].cpu().numpy()
    # generated_images.append(generated_images)

[ 3094  3493  5615 12064   509  5862 11577  5924  4965  8386]
[PosixPath('stylegan3/data/train/cat/pixabay_cat_004185.png'), PosixPath('stylegan3/data/train/cat/pixabay_cat_004021.png'), PosixPath('stylegan3/data/train/wild/flickr_wild_001525.png'), PosixPath('stylegan3/data/train/dog/pixabay_dog_000885.png'), PosixPath('stylegan3/data/train/cat/flickr_cat_000346.png'), PosixPath('stylegan3/data/train/wild/flickr_wild_002737.png'), PosixPath('stylegan3/data/train/dog/pixabay_dog_000903.png'), PosixPath('stylegan3/data/train/wild/pixabay_wild_001203.png'), PosixPath('stylegan3/data/train/cat/pixabay_cat_001620.png'), PosixPath('stylegan3/data/train/wild/flickr_wild_001571.png')]


10it [00:00, 21.22it/s]


In [90]:
import matplotlib.pyplot as plt

In [91]:
# Set up the plot to display images side by side
n_images = len(original_images)
fig, axes = plt.subplots(n_images, 2, figsize=(10, n_images * 3))

# Loop through images and plot them side by side
for i in range(n_images):
    # Plot first image from list1
    axes[i, 0].imshow(original_images[i])
    axes[i, 0].axis('off')  # Hide axis

    # Plot second image from list2
    axes[i, 1].imshow(generated_images[i])
    axes[i, 1].axis('off')  # Hide axis

# Display the plot
# plt.tight_layout()
plt.savefig("1.png")
plt.show()

<IPython.core.display.Javascript object>

## 7. Naive Training

In [ ]:
LR_NAIVE = 1e-3

In [ ]:
cost = MLPLSECost(**cost_config.model_dump())

In [ ]:
model = GMMEOT(
    y_dim=Y_DIM,
    n_potentials=N_POTENTIALS,
    cost=cost,
).to(dtype)

if INIT_BY_SAMPLES:
    model.init_a_by_samples(Y_sampler.sample(N_POTENTIALS))

In [ ]:
# For EMA update
if train_config.ema_update:
    model_copy = GMMEOT(
    y_dim=Y_DIM,
    n_potentials=N_POTENTIALS,
    cost=cost,
).to(dtype)

In [ ]:
D_opt_naive = torch.optim.Adam(model.parameters(), lr=LR_NAIVE)

In [ ]:
if not os.path.exists(OUTPUT_PATH):
    os.makedirs(OUTPUT_PATH, exist_ok=True)

In [ ]:
wandb.init(name=EXP_NAME, config=config)

for step in tqdm(range(train_config.steps_from, train_config.steps_to)):
    # training loop
    D_opt_naive.zero_grad()

    X = usd_sampler.sample(train_config.unpaired_batch_size)
    Y = utd_sampler.sample(train_config.unpaired_batch_size)

    log_w_n = model.log_w_n()
    a_n = model.a_n()
    A_n = model.A_n()

    cond_distr_unpaired = model.get_conditional_distribution(
        X.repeat(train_config.unpaired_batch_size, 1), log_w_n, a_n, A_n
    )
    fwd = cond_distr_unpaired.log_prob(Y.repeat(train_config.unpaired_batch_size, 1))
    D_loss_unpaired = -torch.log(
        torch.mean(torch.exp(fwd.reshape(train_config.unpaired_batch_size, train_config.unpaired_batch_size)), dim=-1)
    ).mean()

    wandb.log({f"Unpaired loss": D_loss_unpaired.item()}, step=step)

    X_paired, Y_paired = pd_train_sampler.sample(train_config.paired_batch_size)

    cond_distr_paired = model.get_conditional_distribution(X_paired, log_w_n, a_n, A_n)
    D_loss_paired = -cond_distr_paired.log_prob(Y_paired).mean()

    wandb.log({f"Paired loss": D_loss_paired.item()}, step=step)

    D_loss = D_loss_unpaired + D_loss_paired
    D_loss.backward()
    D_opt_naive.step()

    if train_config.ema_update:
        update_average(model_copy, model, 0.99)
        model = model_copy
    else:
        model = model

    wandb.log({f"Loss": D_loss}, step=step)
    wandb.log(
        {f"Train paired loss": compute_loss(model, X_paired_train, Y_paired_train, X_paired_train, Y_paired_train)},
        step=step,
    )
    wandb.log(
        {f"Test paired loss": compute_loss(model, X_paired_test, Y_paired_test, X_paired_test, Y_paired_test)},
        step=step,
    )
    wandb.log(
        {f"Test unpaired loss": compute_loss(model, X_unpaired_test, Y_unpaired_test, X_paired_test, Y_paired_test)},
        step=step,
    )

    wandb.log({f"lam_min(A_n)": torch.min(A_n)}, step=step)
    wandb.log({f"lam_max(A_n)": torch.max(A_n)}, step=step)

    if step % train_config.plot_every == 0:
        A_dict = plot_A_parameters(model, log=True)
        B_dict = plot_B_parameters(model.cost, starting_points, log=True)
        if num_starting_points_paired > 0:
            Z_dict = plot_Z_parameters(model, starting_points, starting_points_paired, ending_points_paired, log=True)
        else:
            Z_dict = plot_Z_parameters(model, starting_points, log=True)
        distr_dict = plot_swiss_roll(
            {f"P={P_XY_PAIRED_SAMPLES}, Q={Q_X_UNPAIRED_SAMPLES}, R={R_Y_UNPAIRED_SAMPLES}": model},
            X_sampler,
            Y_sampler,
            X_paired,
            Y_paired,
            starting_points,
            gt_Y_points,
            log=True,
        )
        wandb.log(A_dict | B_dict | Z_dict | distr_dict)

        torch.save(model.state_dict(), os.path.join(OUTPUT_PATH, f"D_{step}.pt"))

torch.save(model.state_dict(), os.path.join(OUTPUT_PATH, f"D_{MAX_STEPS}.pt"))
torch.save(D_opt_paired.state_dict(), os.path.join(OUTPUT_PATH, f"D_opt_paired_{MAX_STEPS}.pt"))
torch.save(D_opt_unpaired.state_dict(), os.path.join(OUTPUT_PATH, f"D_opt_unpaired_{MAX_STEPS}.pt"))

wandb.finish()

## Plotting

In [ ]:
Q_X_unpaired_samples_list = [0, 1024]
R_Y_unpaired_samples_list = [0, 1024]
log_step = 99000

In [ ]:
models_dict = dict()

for i, Q_X_unpaired_samples in enumerate(Q_X_unpaired_samples_list):
    for j, R_Y_unpaired_samples in enumerate(R_Y_unpaired_samples_list):
        model = GMMEOT(
            y_dim=Y_DIM,
            n_potentials=N_POTENTIALS,
            cost=cost,
        ).to(dtype)
        exp_name = EXP_NAME.replace(
            f"Q_X_UNPAIRED_{Q_X_UNPAIRED_SAMPLES}_R_Y_UNPAIRED_{R_Y_UNPAIRED_SAMPLES}_",
            f"Q_X_UNPAIRED_{Q_X_unpaired_samples}_R_Y_UNPAIRED_{R_Y_unpaired_samples}_",
        )
        print(exp_name)
        output_path = "../checkpoints/{}".format(exp_name)
        model.load_state_dict(torch.load(os.path.join(output_path, f"D_{log_step}.pt"), map_location=device))
        title = f"P={P_XY_PAIRED_SAMPLES}, Q={Q_X_unpaired_samples}, R={R_Y_unpaired_samples}"
        models_dict[title] = model

In [ ]:
plot_swiss_roll(
    models_dict,
    X_sampler,
    Y_sampler,
    X_paired_train,
    Y_paired_train,
    starting_points,
    gt_Y_points,
) 